# Week 2 Day 2 (Rewritten for Gemini using LangGraph/LangChain)
We are going to build a simple Agent system for generating cold sales outreach emails using Gemini!
This notebook replaces the `openai-agents` SDK with standard `langchain` and `langchain_openai` which are fully compatible with Gemini via its OpenAI compatibility layer.


In [3]:
import os
import asyncio
from dotenv import load_dotenv

# Load environment variables
load_dotenv(r'c:\Users\abhin\Dropbox\PC\Downloads\projects\.env')

# Setup Gemini as an OpenAI-compatible endpoint
os.environ["OPENAI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")
os.environ["OPENAI_BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gemini-2.5-flash")


c:\Users\abhin\Dropbox\PC\Downloads\projects\.venv\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1


In [4]:
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content

def send_test_email():
    api_key = os.environ.get('SENDGRID_API_KEY')
    if not api_key:
        print("No SendGrid API Key found!")
        return
    sg = sendgrid.SendGridAPIClient(api_key=api_key)
    from_email = Email("abhinavsingh649@gmail.com") 
    to_email = To("abhinavsingh649@gmail.com") 
    content = Content("text/plain", "This is a test email from Gemini!")
    try:
        mail = Mail(from_email, to_email, "Test email", content).get()
        response = sg.client.mail.send.post(request_body=mail)
        print("Email status code:", response.status_code)
    except Exception as e:
        print("Error:", e)

# send_test_email()


### Step 1: Create our Agents

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

instructions1 = "You are a professional, serious sales agent working for ComplAI. Write a cold email for the given request."
instructions2 = "You are a humorous, engaging sales agent working for ComplAI. Write a witty cold email."
instructions3 = "You are a busy, concise sales agent working for ComplAI. Write a short, to-the-point cold email."

def create_agent(instructions):
    prompt = ChatPromptTemplate.from_messages([
        ("system", instructions),
        ("user", "{input}")
    ])
    return prompt | llm | StrOutputParser()

agent1 = create_agent(instructions1)
agent2 = create_agent(instructions2)
agent3 = create_agent(instructions3)


### Step 2: Run in Parallel

In [6]:
async def generate_emails(message):
    results = await asyncio.gather(
        agent1.ainvoke({"input": message}),
        agent2.ainvoke({"input": message}),
        agent3.ainvoke({"input": message})
    )
    return results

emails = await generate_emails("Write a cold sales email addressed to 'Dear CEO'")
for i, email in enumerate(emails):
    print(f"\n--- Email {i+1} ---\n{email}")



--- Email 1 ---
Subject: Proactive Compliance & Enterprise Risk Mitigation for Your Organization

Dear CEO,

At the executive level, balancing strategic growth with robust governance is a constant challenge. Specifically, navigating the escalating complexities of regulatory compliance and managing enterprise risk often consumes significant resources and attention, directly impacting profitability and brand reputation.

My name is [Your Name] and I'm a Sales Agent at ComplAI. We specialize in empowering leaders like yourself to transform how your organization approaches these critical areas. ComplAI is an advanced AI-powered platform designed to provide:

*   **Real-time Visibility:** Gain an instant, comprehensive overview of your compliance posture across all departments and regulatory frameworks.
*   **Proactive Risk Identification:** Leverage predictive analytics to anticipate emerging risks and regulatory changes before they become costly problems.
*   **Operational Efficiency:** 

### Step 3: Pick the Best Email

In [7]:
picker_prompt = ChatPromptTemplate.from_messages([
    ("system", "You pick the best cold sales email from the options. Imagine you are a customer and pick the one you are most likely to respond to. Reply ONLY with the exact text of the selected email. Do not give any explanation."),
    ("user", "Options:\n\n{emails}")
])
picker_agent = picker_prompt | llm | StrOutputParser()

combined_emails = "\n\n---NEXT OPTION---\n\n".join(emails)
best_email = await picker_agent.ainvoke({"emails": combined_emails})

print("\n*** BEST EMAIL ***\n")
print(best_email)



*** BEST EMAIL ***

Subject: Is Your Compliance Department Secretly a Sci-Fi Horror Film? (ComplAI Can Help!)

Dear CEO,

Let's be honest, "compliance" isn't exactly the word that sparks joy. It's more likely to spark existential dread, a sudden craving for strong coffee, or a desperate urge to update your LinkedIn profile for a less regulated industry (like, say, professional napping).

I imagine your inbox is probably a battlefield, so I'll cut straight to the chase: Are you tired of compliance feeling like a relentless game of whack-a-mole, where every mole has a law degree and a penchant for punitive fines? Or worse, a 3 AM cold sweat-inducing nightmare involving spreadsheets, obscure legislative updates, and the looming shadow of an audit?

**What if I told you there's a better way? A way that doesn't involve sacrificing your weekends or turning your brightest minds into regulatory archaeologists?**

Enter **ComplAI** – your new, ridiculously intelligent, and surprisingly witty A

### Step 4: Use a Tool to send

In [9]:
from langchain_core.tools import tool

@tool
def send_email(body: str) -> str:
    """Send out an email with the given body to the sales prospect."""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("abhinavsingh649@gmail.com") 
    to_email = To("abhinavsingh649@gmail.com") 
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    try:
        sg.client.mail.send.post(request_body=mail)
        return "Success"
    except Exception as e:
        return str(e)

llm_with_tools = llm.bind_tools([send_email])


In [10]:
manager_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Sales Manager at ComplAI. You have selected a winning email draft.\n"
               "Now you must use the send_email tool to send exactly the draft text to the prospect.\n"
               "Do not modify the text or add pleasantries."),
    ("user", "Please send the following email draft:\n\n{email_draft}")
])

manager_agent = manager_prompt | llm_with_tools

response = await manager_agent.ainvoke({"email_draft": best_email})

if response.tool_calls:
    print("Manager decided to call tool:", response.tool_calls[0]['name'])
    # Execute the tool
    tool_msg = send_email.invoke(response.tool_calls[0]['args'])
    print("Tool execution result:", tool_msg)
else:
    print("Manager didn't call the tool. Output was:", response.content)


Manager decided to call tool: send_email
Tool execution result: Success
